# Data analysis
## Import libraries and load dataset

In [1]:
import pandas as pd            
import os
import re

# Check the cwd
os.getcwd()

'c:\\Users\\casaz\\Documents\\Simone\\University\\dhdk\\thesis\\evalita_task\\evalita2026-multipride-UniBO-FICLIT'

In [2]:
# Import data from CSV
data = pd.read_csv('data/train_it.csv')
data

,id,text,bio,label,lang
0,it_1231,La destra Italiana pur di non dire che loro od...,Il rispetto per il prossimo qualunque sia il s...,0,it
1,it_1713,"""Presupporre che tutti i bisessuali non sono m...",𝓕𝓲𝓵𝓵𝓮𝓭 𝔀𝓲𝓽𝓱 𝓯𝓾𝓻𝔂 𝓪𝓷𝓭 𝓼𝓽𝓪𝓻𝓻𝔂 𝓮𝔂𝓮𝓭,0,it
2,it_1474,"Se i diritti devono essere uguali, voglio che ...",User Experience Designer URL,0,it
3,it_58,che poi molti uomini trans subiscono lesbofobi...,"no matter where i go, you're there …",0,it
4,it_511,Che poi è l’etero medio come Pio e Amedeo che ...,T'appartengo ed io ci tengo \nE se prometto po...,0,it
...,...,...,...,...,...
1081,it_1340,Io mi sono rotto i coglioni di tutte queste po...,"Se incapaci di scherzare, se il resto del mond...",0,it
1082,it_595,"come una donna cis non può subire transfobia, ...",알잖아 널 가만히 둘 수 없는 걸,0,it
1083,it_844,Ehi comunità @USER questo giornalista di repub...,NaN,0,it
1084,it_1216,"@USER tom felton non è transfobico, ma ha anch...",NaN,0,it


## Basic stats

In [3]:
# some basic stats
print(len(data))
print(data['label'].value_counts())
print(data.info())

1086
0    879
1    207
Name: label, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1086 entries, 0 to 1085
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      1086 non-null   object
 1   text    1086 non-null   object
 2   bio     1000 non-null   object
 3   label   1086 non-null   int64 
 4   lang    1086 non-null   object
dtypes: int64(1), object(4)
memory usage: 42.5+ KB
None


In [4]:
# Print average length of texts in tokens
data['text_length'] = data['text'].apply(lambda x: len(x.split()))
print(data['text_length'].mean())

# Print average length of biographies in tokens (if available, otherwise do not count them)
data['bio_length'] = data['bio'].apply(lambda x: len(x.split()) if pd.notnull(x) else None)
print(data['bio_length'].mean())

34.06813996316759
13.302


## Lexical and Semantic Analysis
### Tokenization and Cleaning

In [5]:
# Check for user mentions
data["user_mention"] = data["text"].apply(lambda x: re.findall(r"\s@\w+", x))

data[data["user_mention"].apply(lambda x: len(x) > 0)]

,id,text,bio,label,lang,text_length,bio_length,user_mention
6,it_1798,@USER @USER comunque la tua transfobia traspar...,"influenzer \npansexual, genderqueer and polyam...",0,it,32,24.0,[ @USER]
17,it_1104,@USER @USER io ci vengo ai vostri raduni Gay P...,NaN,0,it,51,NaN,[ @USER]
18,it_1717,@USER @USER perché il femminismo si occupa di ...,piangente come i salici || radfem,0,it,40,6.0,[ @USER]
33,it_996,@USER @USER @USER “Frocio di merda” è omofobia...,NaN,0,it,30,NaN,"[ @USER, @USER]"
41,it_212,@USER @USER @USER Essere gay o trans non si in...,"Passionate about Queen, Star Trek, Apple, Culm...",0,it,51,20.0,"[ @USER, @USER]"
...,...,...,...,...,...,...,...,...
1037,it_1,@USER @USER @USER @USER @USER Basta già il vos...,Art. 52 della Costituzione: “La difesa della P...,0,it,32,14.0,"[ @USER, @USER, @USER, @USER]"
1051,it_280,@USER @USER @USER Esatto quindi la retorica di...,📈 DIGITAL RISK STRATEGIST enhancing #Digital #...,0,it,34,12.0,"[ @USER, @USER]"
1053,it_1173,@USER @USER Dopo la sfila dei trans e dei LGBT...,#̷̧̢̼̤͈̼̫̋̾Q̸̼͔͕̗͚̟̉̄̄͝a̷̞̗̺̐̔̚ṇ̶̨̢̦͚͚̜͇̯̉̓̑̉̏...,0,it,46,2.0,[ @USER]
1059,it_1381,@USER @USER E xenocomunista frocialista oltran...,Ho varcato le sogne del male e ho osato dire c...,1,it,7,24.0,[ @USER]


In [6]:
# Check for urls
data["urls"] = data["text"].apply(lambda x: re.findall(r"www", x))
data[data["urls"].apply(lambda x: len(x) > 0)]

,id,text,bio,label,lang,text_length,bio_length,user_mention,urls


There are no urls in the tweets. Since all user mentions are hidden for privacy, we can remove them without losing meaning.

In [16]:
# Function to clean the tweets
def clean_text(text: str) -> str:
    # Remove user mentions
    text = re.sub(r"@\w+", "", text)
    
    # Remove useless extra space
    text_clean = re.sub(r"\s+", " ", text).strip()

    return text_clean

In [17]:
# Drop useless columns
data = data.drop(columns=["text_length", "bio_length", "user_mention", "urls"])

KeyError: "['text_length', 'bio_length', 'user_mention', 'urls'] not found in axis"

In [18]:
data["text_clean"] = data["text"].apply(lambda x: clean_text(x))

In [19]:
data

,id,text,bio,label,lang,text_clean
0,it_1231,La destra Italiana pur di non dire che loro od...,Il rispetto per il prossimo qualunque sia il s...,0,it,La destra Italiana pur di non dire che loro od...
1,it_1713,"""Presupporre che tutti i bisessuali non sono m...",𝓕𝓲𝓵𝓵𝓮𝓭 𝔀𝓲𝓽𝓱 𝓯𝓾𝓻𝔂 𝓪𝓷𝓭 𝓼𝓽𝓪𝓻𝓻𝔂 𝓮𝔂𝓮𝓭,0,it,"""Presupporre che tutti i bisessuali non sono m..."
2,it_1474,"Se i diritti devono essere uguali, voglio che ...",User Experience Designer URL,0,it,"Se i diritti devono essere uguali, voglio che ..."
3,it_58,che poi molti uomini trans subiscono lesbofobi...,"no matter where i go, you're there …",0,it,che poi molti uomini trans subiscono lesbofobi...
4,it_511,Che poi è l’etero medio come Pio e Amedeo che ...,T'appartengo ed io ci tengo \nE se prometto po...,0,it,Che poi è l’etero medio come Pio e Amedeo che ...
...,...,...,...,...,...,...
1081,it_1340,Io mi sono rotto i coglioni di tutte queste po...,"Se incapaci di scherzare, se il resto del mond...",0,it,Io mi sono rotto i coglioni di tutte queste po...
1082,it_595,"come una donna cis non può subire transfobia, ...",알잖아 널 가만히 둘 수 없는 걸,0,it,"come una donna cis non può subire transfobia, ..."
1083,it_844,Ehi comunità @USER questo giornalista di repub...,NaN,0,it,Ehi comunità questo giornalista di repubblica ...
1084,it_1216,"@USER tom felton non è transfobico, ma ha anch...",NaN,0,it,"tom felton non è transfobico, ma ha anche dife..."


### Frequency Analysis

### Characteristic Term Analysis

In [ ]:
# Import hurtlex data from TSV
hurtlex = pd.read_csv('data/hurtlex_IT.tsv', sep='\t')
                                                                        
hurtlex.head()

In [ ]:
homophobic_terms = hurtlex[hurtlex['category'] == "om"]

homophobic_terms

In [ ]:
# create a set of homophobic terms for faster lookup
homophobic_set = set(homophobic_terms['lemma'].str.lower())

# add also feminine forms (simple heuristic: add 'a' at the end if it ends with 'o')
homophobic_set.update({term[:-1] + 'a' for term in homophobic_set if term.endswith('o')})

print(len(homophobic_set))

In [ ]:
# Check for homophobic terms in the tweets end return a column with the terms found
def contains_homophobic_term(text):
    words = text.lower().split()
    for word in words:
        if word in homophobic_set:
            return word
    return None
data['contains_homophobic'] = data['text'].apply(contains_homophobic_term)
data[['text', 'contains_homophobic']]


In [ ]:
# Drop rows with None values in 'contains_homophobic' column
data_with_homophobic = data.dropna(subset=['contains_homophobic'])

data_with_homophobic[['text', 'contains_homophobic']]

In [ ]:
data_with_homophobic[data_with_homophobic['contains_homophobic'] == 'queer'][['text', 'contains_homophobic', 'label']]